<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/LGBM_%E5%9D%87%E7%B7%9A%E7%B3%BE%E7%B5%90_60MA_%E5%A4%96%E8%B3%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gc
import time
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from sklearn.model_selection import TimeSeriesSplit

# 嘗試載入 Colab 專用套件
try:
    from google.colab import files

    HAS_COLAB = True
except ImportError:
    HAS_COLAB = False

warnings.filterwarnings("ignore")

# 設定預測信心絕對門檻（大於等於 60% 才輸出）
CONFIDENCE_THRESHOLD = 0.60

stock_dict = {
    "0050.TW": "元大台灣50",
    "0056.TW": "元大高股息",
    "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式",
    "SPCX": "SPACs ETF",
    "SOXX": "iShares半導體ETF",
    "SMH": "VanEck半導體ETF",
    "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet",
    "META": "Meta",
    "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達",
    "TSM": "台積電 ADR",
    "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐",
    "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生",
    "ASML": "ASML 艾司摩爾",
    "AMAT": "Applied Materials 應用材料",
    "LRCX": "Lam Research 柯林研發",
    "KLAC": "KLA 科磊",
    "AMD": "AMD 超微",
    "AVGO": "Broadcom 博通",
    "QCOM": "Qualcomm 高通",
    "INTC": "Intel 英特爾",
    "MU": "Micron 鎂光",
    "TXN": "Texas Instruments 德州儀器",
    "ARM": "ARM 晶心/安謀",
    "MRVL": "Marvell 邁威爾",
    "ADI": "Analog Devices 亞德諾",
    "MPWR": "Monolithic Power 芯源系統",
    "ON": "ON Semiconductor 安森美",
    "SWKS": "Skyworks 思佳訊",
    "QRVO": "Qorvo 威訊",
    "TER": "Teradyne 泰瑞達",
    "MKSI": "MKS Instruments",
    "PANW": "Palo Alto Networks",
    "CRWD": "CrowdStrike",
    "FTNT": "Fortinet",
    "NET": "Cloudflare",
    "ZS": "Zscaler",
    "OKTA": "Okta",
    "S": "SentinelOne",
    "GEN": "Gen Digital",
    "RPD": "Rapid7",
    "CBRS": "CyberArk",
    "2471.TW": "資通",
    "2480.TW": "敦陽科",
    "3029.TW": "零壹",
    "6214.TW": "精誠",
    "3130.TW": "一零四",
    "2427.TW": "三商電",
    "3027.TW": "盛達",
    "5203.TW": "訊連",
    "5471.TW": "松翰",
    "5410.TWO": "國統",
    "6183.TW": "關貿",
    "6203.TWO": "海韻電",
    "6210.TWO": "慶生",
    "6593.TWO": "台灣銘板",
    "6689.TW": "伊雲谷",
    "6690.TWO": "安碁資訊",
    "6752.TWO": "睿嘉",
    "6763.TWO": "綠界科技",
    "6865.TWO": "偉康科技",
    "6874.TWO": "倍力",
    "6928.TW": "全達",
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "2353.TW": "宏碁",
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    "3035.TW": "智原",
    "6643.TWO": "M31",
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創唯",
    "6756.TW": "威鋒電子",
    "2342.TW": "茂矽",
    "6770.TW": "力積電",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "钛昇",
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    "2404.TW": "漢唐",
    "1773.TW": "勝一",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "5536.TWO": "聖暉*",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜晶",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    "6715.TW": "嘉基",
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    "8043.TWO": "蜜望實",
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    "2395.TW": "研華",
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    "3481.TW": "群創",
    "2409.TW": "友達",
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "2393.TW": "億光",
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    "2603.TW": "長榮",
    "2609.TW": "陽明",
    "2615.TW": "萬海",
    "2605.TW": "新興",
    "2606.TW": "裕民",
    "2612.TW": "中航",
    "2617.TW": "台航",
    "2637.TW": "慧洋-KY",
    "2641.TWO": "正德",
    "5608.TW": "四維航",
    "2610.TW": "華航",
    "2618.TW": "長榮航",
    "2630.TW": "亞航",
    "5603.TWO": "陸海",
    "2607.TW": "榮運",
    "2608.TW": "嘉里大榮",
    "2611.TW": "志信",
    "2613.TW": "中櫃",
    "2636.TW": "台驊投控",
    "2642.TW": "宅配通",
    "2633.TW": "台灣高鐵",
    "5607.TW": "遠雄港",
    "5609.TWO": "中菲行",
    "8367.TW": "建新國際",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "1303.TW": "南亞",
    "2465.TW": "麗臺",
    "8163.TW": "達方",
    "3042.TW": "晶技",
    "8182.TWO": "加高",
    "3229.TW": "泰藝",
    "3308.TW": "聯傑",
    "6284.TWO": "佳邦",
    "2484.TW": "希華",
    "8088.TWO": "華信科",
}


# ==========================================
# 優化版：批次預先抓取外資資料的快取函數
# ==========================================
def fetch_twse_foreign_bulk(date_str):  # 格式: YYYYMMDD
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?date={date_str}&selectType=ALLBUT0999&response=json"
    try:
        res = requests.get(
            url,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
                )
            },
            timeout=5,
        )
        if res.status_code == 200:
            js = res.json()
            if js.get("stat") == "OK":
                df = pd.DataFrame(js["data"], columns=js["fields"])
                return df
    except Exception:
        pass
    return None


def fetch_tpex_foreign_bulk(date_str_slash):  # 格式: YYYY/MM/DD
    url = f"https://www.tpex.org.tw/www/zh-tw/insti/qfiiStat?type=Daily&date={date_str_slash}&searchType=buy&id=&response=json"
    try:
        res = requests.get(
            url,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
                )
            },
            timeout=5,
        )
        if res.status_code == 200:
            js = res.json()
            if "tables" in js and len(js["tables"]) > 0:
                t = js["tables"][0]
                df = pd.DataFrame(t["data"], columns=t["fields"])
                return df
    except Exception:
        pass
    return None


def sanitize_df(df):
    if df is None or df.empty:
        return None
    d = df.copy()

    if d.index.tz is not None:
        d.index = d.index.tz_localize(None)

    if isinstance(d.columns, pd.MultiIndex):
        for level in range(d.columns.nlevels):
            col_names = [
                str(c).strip().title() for c in d.columns.get_level_values(level)
            ]
            if "Close" in col_names or "Adj Close" in col_names:
                d.columns = d.columns.get_level_values(level)
                break
        else:
            d.columns = d.columns.get_level_values(-1)

    col_map = {}
    for c in d.columns:
        c_str = str(c).strip().title()
        if "Adj Close" in c_str:
            col_map[c] = "Close"
        elif "Close" in c_str and "Close" not in col_map.values():
            col_map[c] = "Close"
        elif "Open" in c_str:
            col_map[c] = "Open"
        elif "High" in c_str:
            col_map[c] = "High"
        elif "Low" in c_str:
            col_map[c] = "Low"
        elif "Volume" in c_str:
            col_map[c] = "Volume"

    d = d.rename(columns=col_map)
    needed = ["Open", "Close", "High", "Low", "Volume"]

    if not all(k in d.columns for k in needed):
        return None

    return d[needed].dropna(subset=["Close"])


def compute_market_features(market_df):
    m = sanitize_df(market_df)
    if m is None:
        raise ValueError("大盤數據處理失敗！")

    m_returns = m["Close"].pct_change()
    m_vol_20 = m_returns.rolling(20).std() * np.sqrt(252)
    m_ret_20 = m["Close"].pct_change(20)
    m_ma20 = m["Close"].rolling(20).mean()
    m_ma_dist = (m["Close"] - m_ma20) / (m_ma20 + 1e-6)

    market_feats = pd.DataFrame(
        {
            "Market_Vol_20": m_vol_20,
            "Market_Ret_20": m_ret_20,
            "Market_MA_Dist": m_ma_dist,
            "Market_Close": m["Close"],
        },
        index=m.index,
    )
    return market_feats


def get_market_data():
    try:
        tw_m = yf.download(
            "^TWII", period="3y", progress=False, auto_adjust=True
        )
        tw_feats = compute_market_features(tw_m)
    except Exception:
        tw_m = yf.download(
            "0050.TW", period="3y", progress=False, auto_adjust=True
        )
        tw_feats = compute_market_features(tw_m)

    try:
        us_m = yf.download(
            "^GSPC", period="3y", progress=False, auto_adjust=True
        )
        us_feats = compute_market_features(us_m)
    except Exception:
        us_m = yf.download(
            "SPY", period="3y", progress=False, auto_adjust=True
        )
        us_feats = compute_market_features(us_m)

    return tw_feats, us_feats


def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))


print("步驟零：下載雙市場大盤行情數據並準備個股資料庫...")
tw_market_feats, us_market_feats = get_market_data()

print("正在批次預載最近交易日之外資籌碼資料...")
dummy_df = yf.download("2330.TW", period="1y", progress=False, auto_adjust=True)
dummy_df = sanitize_df(dummy_df)
recent_dates = dummy_df.index[-30:]

twse_foreign_cache = {}
tpex_foreign_cache = {}

for r_date in recent_dates:
    date_str = r_date.strftime("%Y%m%d")
    date_str_slash = r_date.strftime("%Y/%m/%d")

    df_twse = fetch_twse_foreign_bulk(date_str)
    if df_twse is not None and not df_twse.empty:
        mapping = {}
        for _, row in df_twse.iterrows():
            code_str = str(row.iloc[0]).strip()
            try:
                val = float(str(row.iloc[4]).replace(",", ""))
                mapping[code_str] = val
            except Exception:
                pass
        twse_foreign_cache[r_date] = mapping

    df_tpex = fetch_tpex_foreign_bulk(date_str_slash)
    if df_tpex is not None and not df_tpex.empty:
        mapping = {}
        for _, row in df_tpex.iterrows():
            code_str = str(row.iloc[0]).strip()
            try:
                val = float(str(row.iloc[4]).replace(",", ""))
                mapping[code_str] = val
            except Exception:
                pass
        tpex_foreign_cache[r_date] = mapping

print("外資籌碼快取載入完成，開始進行個股特徵計算...")


def compute_features(df, market_feats, is_tw_stock=False, ticker=None):
    d = sanitize_df(df)
    if d is None or len(d) < 200:
        return None, []

    x = np.arange(5)
    x_dev = x - x.mean()
    x_var = (x_dev**2).sum()

    def calc_slope_5(y):
        return (x_dev * (y - y.mean())).sum() / x_var

    d["Close_Slope"] = (
        d["Close"].rolling(5).apply(calc_slope_5, raw=True) / (d["Close"] + 1e-6)
    )

    tr = pd.concat(
        [
            d["High"] - d["Low"],
            np.abs(d["High"] - d["Close"].shift(1)),
            np.abs(d["Low"] - d["Close"].shift(1)),
        ],
        axis=1,
    ).max(axis=1)

    d["NATR"] = tr.rolling(14).mean() / (d["Close"] + 1e-6)
    ma20 = d["Close"].rolling(20).mean()
    ma60 = d["Close"].rolling(60).mean()

    d["BB_Bandwidth"] = (
        ma20
        + 2 * d["Close"].rolling(20).std()
        - (ma20 - 2 * d["Close"].rolling(20).std())
    ) / (ma20 + 1e-6)
    d["BIAS_5"] = (
        d["Close"] - d["Close"].rolling(5).mean()
    ) / (d["Close"].rolling(5).mean() + 1e-6)

    ema12 = d["Close"].ewm(span=12, adjust=False).mean()
    ema26 = d["Close"].ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    d["MACD_Hist"] = (macd_line - signal_line) / (d["Close"] + 1e-6)
    d["MACD_Hist_Slope"] = d["MACD_Hist"].diff(3)

    vol_ratio = d["Volume"] / (d["Volume"].rolling(5).mean() + 1e-6)
    d["Volume_Explosion"] = np.clip(vol_ratio, 0, 10)

    turnover = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)
    d["Turnover_Rate"] = np.clip(turnover, 0, 10)

    k_range = d["High"] - d["Low"]
    d["Body_Ratio"] = (d["Close"] - d["Open"]).abs() / (k_range + 1e-6)
    d["RSI_14"] = compute_rsi(d["Close"], 14) / 100.0
    d["RSI_Slope"] = d["RSI_14"].diff(3)

    d["Foreign_Net_Vol_Ratio"] = 0.0
    d["Foreign_Net_MA5"] = 0.0

    if is_tw_stock and ticker:
        pure_ticker = ticker.split(".")[0]
        for r_date in d.index:
            foreign_net = 0.0
            if ticker.endswith(".TW") and r_date in twse_foreign_cache:
                day_map = twse_foreign_cache[r_date]
                for k, v in day_map.items():
                    if pure_ticker in k:
                        foreign_net = v
                        break
            elif ticker.endswith(".TWO") and r_date in tpex_foreign_cache:
                day_map = tpex_foreign_cache[r_date]
                for k, v in day_map.items():
                    if pure_ticker in k:
                        foreign_net = v
                        break

            vol = d.loc[r_date, "Volume"]
            d.loc[r_date, "Foreign_Net_Vol_Ratio"] = foreign_net / (vol + 1e-6)

        d["Foreign_Net_MA5"] = (
            d["Foreign_Net_Vol_Ratio"].rolling(5).mean().fillna(0)
        )

    m_reindexed = market_feats.reindex(d.index).ffill()
    if is_tw_stock:
        m_pct = m_reindexed["Market_Close"].shift(1).pct_change(5)
    else:
        m_pct = m_reindexed["Market_Close"].pct_change(5)

    d["Alpha_5d"] = d["Close"].pct_change(5) - m_pct
    d["Market_Vol_20"] = m_reindexed["Market_Vol_20"]
    d["Market_Ret_20"] = m_reindexed["Market_Ret_20"]
    d["Market_MA_Dist"] = m_reindexed["Market_MA_Dist"]

    ma5 = d["Close"].rolling(5).mean()
    ma_tangle = (ma5 - ma20).abs() / (d["Close"] + 1e-6) <= 0.04
    turnover_amount = (d["Close"] * d["Volume"]).rolling(5).mean()
    min_turnover = 30_000_000 if is_tw_stock else 2_000_000
    liquidity_ok = turnover_amount >= min_turnover

    above_ma60 = d["Close"] > ma60
    d["Filter_Pass"] = ma_tangle & liquidity_ok & above_ma60

    # ==========================================
    # 修改處：10個交易日內「最高漲幅 >= 8%」且「未跌破 -4%」
    # ==========================================
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=10)
    future_max = d["Close"].shift(-1).rolling(window=indexer, min_periods=10).max()
    future_min = d["Close"].shift(-1).rolling(window=indexer, min_periods=10).min()

    hit_profit = (future_max - d["Close"]) / (d["Close"] + 1e-6) >= 0.08
    hit_stop_loss = (future_min - d["Close"]) / (d["Close"] + 1e-6) <= -0.04

    d["Target"] = (hit_profit & (~hit_stop_loss)).astype(float)
    d.iloc[-10:, d.columns.get_loc("Target")] = np.nan

    float_cols = d.select_dtypes(include=["float64"]).columns
    d[float_cols] = d[float_cols].astype("float32")

    feature_cols = [
        "Close_Slope",
        "NATR",
        "BB_Bandwidth",
        "BIAS_5",
        "Volume_Explosion",
        "Turnover_Rate",
        "Body_Ratio",
        "RSI_14",
        "RSI_Slope",
        "MACD_Hist",
        "MACD_Hist_Slope",
        "Alpha_5d",
        "Market_Vol_20",
        "Market_Ret_20",
        "Market_MA_Dist",
        "Foreign_Net_Vol_Ratio",
        "Foreign_Net_MA5",
    ]
    return d, feature_cols


all_dfs = []
feature_cols = []
tickers = list(stock_dict.keys())
batch_size = 15

for i in range(0, len(tickers), batch_size):
    sub_tickers = tickers[i : i + batch_size]
    batch_data = None
    for attempt in range(3):
        try:
            batch_data = yf.download(
                sub_tickers,
                period="1y",
                progress=False,
                group_by="ticker",
                threads=False,
                auto_adjust=True,
            )
            if batch_data is not None and not batch_data.empty:
                break
        except Exception:
            time.sleep(1)

    if batch_data is None or batch_data.empty:
        continue

    for ticker in sub_tickers:
        try:
            name = stock_dict[ticker]
            if len(sub_tickers) == 1:
                df = batch_data.copy()
            else:
                if isinstance(batch_data.columns, pd.MultiIndex):
                    tickers_in_data = batch_data.columns.get_level_values(0).unique()
                    if ticker not in tickers_in_data:
                        continue
                    df = batch_data[ticker].dropna(how="all")
                else:
                    df = batch_data.dropna(how="all")

            is_tw = ticker.endswith(".TW") or ticker.endswith(".TWO")
            m_feats = tw_market_feats if is_tw else us_market_feats

            df_feat, f_cols = compute_features(
                df, m_feats, is_tw_stock=is_tw, ticker=ticker
            )

            if df_feat is None:
                continue

            feature_cols = f_cols
            df_feat["Ticker"] = ticker
            df_feat["Stock_Name"] = name
            all_dfs.append(df_feat)
        except Exception:
            continue

    gc.collect()

if not all_dfs:
    raise ValueError("未能順利取得個股數據，請檢查網路連線後重試。")

panel_df = pd.concat(all_dfs).sort_index()

print("步驟一：模型滾動交叉驗證與歷史勝率計算 (Purged Out-of-Fold)...")
unique_dates = pd.Series(panel_df.index.unique()).sort_values().reset_index(drop=True)
latest_date = unique_dates.iloc[-1]
cutoff_date = unique_dates.iloc[-6]

panel_clean = panel_df.dropna(subset=feature_cols)

hist_data = panel_clean[
    (panel_clean.index <= cutoff_date) & (panel_clean["Target"].notnull())
]
hist_filtered = hist_data[hist_data["Filter_Pass"]].copy()

unique_hist_dates = hist_filtered.index.unique().sort_values()

num_unique_dates = len(unique_hist_dates)
if num_unique_dates >= 6:
    n_splits = min(5, num_unique_dates - 1)
    tscv = TimeSeriesSplit(n_splits=n_splits)
    hist_filtered["OOF_Prob"] = np.nan

    for train_date_idx, val_date_idx in tscv.split(unique_hist_dates):
        train_dates = unique_hist_dates[train_date_idx]
        val_dates = unique_hist_dates[val_date_idx]

        min_val_date = val_dates.min()
        prior_dates = unique_hist_dates[unique_hist_dates < min_val_date]

        purge_limit_date = prior_dates[-5] if len(prior_dates) >= 5 else min_val_date

        train_dates_purged = train_dates[train_dates < purge_limit_date]
        if len(train_dates_purged) == 0:
            train_dates_purged = train_dates

        train_mask = hist_filtered.index.isin(train_dates_purged)
        val_mask = hist_filtered.index.isin(val_dates)

        X_tr, y_tr = (
            hist_filtered.loc[train_mask, feature_cols],
            hist_filtered.loc[train_mask, "Target"],
        )
        X_va = hist_filtered.loc[val_mask, feature_cols]

        if len(X_tr) == 0 or len(X_va) == 0 or len(np.unique(y_tr)) < 2:
            continue

        cv_model = lgb.LGBMClassifier(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            verbose=-1,
            random_state=42,
        )
        cv_model.fit(X_tr, y_tr)

        hist_filtered.loc[val_mask, "OOF_Prob"] = cv_model.predict_proba(X_va)[
            :, 1
        ]
else:
    hist_filtered["OOF_Prob"] = np.nan

print("步驟二：使用全量歷史訓練全市場 LightGBM 模型並進行最新預測...")
X_hist_full = hist_filtered[feature_cols]
y_hist_full = hist_filtered["Target"]

if len(np.unique(y_hist_full)) < 2:
    raise ValueError("歷史資料標籤類別不足 2 種，無法訓練分類模型。")

full_model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    verbose=-1,
    random_state=42,
)
full_model.fit(X_hist_full, y_hist_full)

latest_df = panel_clean[
    (panel_clean.index == latest_date) & panel_clean["Filter_Pass"]
].copy()

if not latest_df.empty:
    latest_df["raw_prob"] = full_model.predict_proba(latest_df[feature_cols])[
        :, 1
    ]

    confident_df = latest_df[latest_df["raw_prob"] >= CONFIDENCE_THRESHOLD]
    top_targets = confident_df.sort_values(by="raw_prob", ascending=False).head(15)

    all_high_conf = hist_filtered[hist_filtered["OOF_Prob"] >= 0.5]
    market_baseline_p = (
        (all_high_conf["Target"] == 1).mean() if len(all_high_conf) > 0 else 0.5
    )

    print(
        f"\n[全市場基準] 歷史高信心平均勝率: {market_baseline_p*100:.2f}%\n"
    )

    if top_targets.empty:
        print(f"今日全市場無標的符合漲升機率 >= {CONFIDENCE_THRESHOLD*100:.0f}% 門檻！")
    else:
        comparison_results = []
        for _, row in top_targets.iterrows():
            s_name = row["Stock_Name"]
            comparison_results.append(
                {
                    "股票名稱": s_name,
                    "股票代號": row["Ticker"],
                    "預測日期": latest_date.strftime("%Y-%m-%d"),
                    "模型預測漲升機率": f"{row['raw_prob']*100:.2f}%",
                    "外資買賣超比": f"{row['Foreign_Net_Vol_Ratio']*100:.2f}%",
                }
            )

        result_df = pd.DataFrame(comparison_results)
        print(result_df.to_string(index=False))

        file_name = "top_confident_with_foreign_stats.xlsx"
        result_df.to_excel(file_name, index=False)
        print(f"\n報表 {file_name} 已成功產生！")

        if HAS_COLAB:
            files.download(file_name)
else:
    print("最新日期無符合高流動性與技術面條件之標的。")
